# Introduction

In the realm of Natural Language Processing (NLP), ensuring that textual data is in a consistent and standardized format is crucial for effective analysis and modeling. Text normalization involves a series of preprocessing tasks aimed at enhancing the quality and uniformity of textual data.

Text normalization, is a technique and algorithm to transform raw text into a more structured and analyzable form. Byte Pair Encoding (BPE) algorithm, is a popular method for subword tokenization which is key component for Transformers.




#  Byte Pair Encoding (BPE) algorithm


In [ ]:
import string


In this notebook I am using "Hamlet" by William Shakespeare. "Hamlet" is a masterpiece of English literature, celebrated for its profound themes, intricate characters, and rich linguistic tapestry.
Run this cell to load the "Hamlet" text data.


In [ ]:
def read_data(file_path):
    # Open the file in read mode ('r')
    with open(file_path, 'r') as file:
        # Read the entire file content
        return file.read()

text = read_data('data/hamlet.txt')


Text Preparation : Removing punctuation and converting to lower case:

In [ ]:
def clean_and_normalize_text(text):
    """
    Clean and normalize the input text by converting it to lowercase
    and removing punctuation.

    Parameters:
    - text (str): The input text to be cleaned.

    Returns:
    - str: The cleaned and normalized text.
    """
    text = text.lower()
    translator = str.maketrans("", "", string.punctuation)
    cleaned_text = text.translate(translator)
    return cleaned_text

cleaned_text = clean_and_normalize_text(text)



Create a function that computes token frequencies and establishes the initial vocabulary:


In [ ]:
def calculate_token_frequencies_and_vocabulary(text):
    """
    Calculate token frequencies and build a vocabulary from the input text.

    Parameters:
    - text (str): The input text.

    Returns:
    - tuple: A tuple containing a dictionary of token frequencies and a set of vocabulary.
    """
    # Create a dictionary to store token frequencies
    token_frequency = {}
    vocabulary = set()
    ## START YOU CODE HERE
    pair_count = 2
    words = list(filter(lambda word: not any(char.isdigit()
                                         for char in word), text.split()))

    for word in words:
        letters = []
        for letter in word:
            letters.append(letter)
            vocabulary.add(letter)

        corpus_word = " ".join(letters)
        corpus_word += " _"
        if corpus_word not in token_frequency:
            token_frequency[corpus_word] = 1
        else:
            token_frequency[corpus_word] += 1

     ## END

    vocabulary.add("_")
    return token_frequency, vocabulary

data, vocabulary = calculate_token_frequencies_and_vocabulary(cleaned_text)


Run the below cell to test your function:

In [ ]:
test_data, test_vocabulary = calculate_token_frequencies_and_vocabulary("low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new")
assert test_data == {'l o w _': 5, 'l o w e s t _': 2, 'n e w e r _': 6, 'w i d e r _': 3, 'n e w _': 2}, "Test failed!"
assert test_vocabulary == {'_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w'}, "Test failed!"
print("Success!")


Success!



Create a function that identifies the most frequent pair of adjacent symbols in the  data:


In [ ]:
def calculate_most_frequent_symbol_pair(data):
    """
    Calculate most frequent pair of adjacent symbols in the given data.

    Parameters:
    - data (dict): A dictionary where keys are words and values are their frequencies.

    Returns:
    - tuple: A tuple containing the most frequent pair of symbols and its frequency.
    """
    best_pair = ()
    frequency = 0
    ## START YOU CODE HERE
    pair_dict = {}
    count = 0

    for word, freq in data.items():
        count = 0
        letters = word.split()
        new_word_len = len(letters)

        for idx in range(new_word_len - 1):
            first_item = letters[idx]
            second_item = letters[idx+1]
            key = (first_item, second_item)

            pattern = ' '.join(key)
            if pattern in word:

                if key not in pair_dict:
                    pair_dict[key] = freq
                else:
                    pair_dict[key] += freq
            count += 1

    best_pair = max(pair_dict, key=pair_dict.get)
    frequency =  pair_dict[best_pair]
    ## END

    return best_pair, frequency

Run the below cell to test your function:

In [ ]:
assert calculate_most_frequent_symbol_pair(test_data) == (('e', 'r'), 9), "Test failed!"
print("Success!")


Success!



Create a function that merges a given symbol and update the data and vocabulary:


In [ ]:
def merge_symbol(pair, data, vocabulary):
    """
    Merge the specified symbol pair into a single symbol and update the data and vocabulary.

    Parameters:
    - pair (tuple): The symbol pair to be merged.
    - data (dict): A dictionary where keys are words, and values are their frequencies.
    - vocabulary (set): A set containing the vocabulary of symbols.

    Returns:
    - tuple: A tuple containing the updated data, updated vocabulary, and the newly created symbol.
    """

    # Merge occurrences of the pair into a single symbol
    updated_data = {}
    ## START YOU CODE HERE
    updated_vocabulary = vocabulary
    new_symbol = ("").join(pair)
    search_chars = ' '.join(pair)

    def merge(word, subword, replacement, updated_data, updated_vocab):
        new_word = word.replace(subword, replacement)
        freq = data[word]
        updated_data[new_word] = freq
        updated_vocab.add(new_symbol)

    for word, freq in data.items():
        if word == search_chars or (" " + search_chars + " " in word):
            merge(word, search_chars, new_symbol, updated_data, updated_vocabulary)
        elif word.startswith(search_chars + " ") or word.endswith(" " + search_chars):
            merge(word, search_chars, new_symbol, updated_data, updated_vocabulary)
        else:
            updated_data[word] = data[word]
    ## END
    return updated_data, updated_vocabulary, new_symbol

Run the below cell to test your function:

In [ ]:
assert merge_symbol(('e', 'r'), test_data, test_vocabulary) == ({'l o w _': 5, 'l o w e s t _': 2, 'n e w er _': 6,
                                                                 'w i d er _': 3, 'n e w _': 2},
                                                                {'_', 'd', 'e', 'er', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w'},
                                                                'er'), "Test failed!"

print("Success!")

assert merge_symbol(('e', 'r'), {'de ri_': 5, 'n e w e r _': 6},
                    {'_', 'd', 'e', 'i', 'n', 'r',}) == ({'de ri_': 5, 'n e w er _': 6},
                                                         {'_', 'd', 'e', 'er', 'i', 'n', 'r'}, 'er'), "Test failed!"
print("Success!")

Success!
Success!



Create the BPE function that will run a number of merges:


In [ ]:
def bpe(data, vocabulary, num_merges):
    """
    Apply Byte Pair Encoding (BPE) to the given data for a specified number of merges.

    Parameters:
    - data (dict): A dictionary where keys are words and values are their frequencies.
    - vocabulary (set): A set containing the initial vocabulary of symbols.
    - num_merges (int): The number of merges to perform.

    Returns:
    - tuple: A tuple containing the updated data, updated vocabulary, and a list of merge rules (ranked).
    """
    merge_rules = []
    updated_data = data
    updated_vocabulary = vocabulary
    ## START YOU CODE HERE
    for n in range(num_merges):
        best_pair, frequency = calculate_most_frequent_symbol_pair(updated_data)
        updated_data, updated_vocabulary, new_symbol = merge_symbol(best_pair, updated_data, updated_vocabulary)

        left = ' '.join(best_pair)
        merge_rules.append((left, new_symbol))
    ## END
    return updated_data, updated_vocabulary, merge_rules

In [ ]:
updated_test_data, updated_test_vocabulary, test_merge_rules = bpe(test_data, test_vocabulary, 8)
assert updated_test_data == {'low_': 5, 'low e s t _': 2, 'newer_': 6, 'w i d er_': 3, 'new _': 2}, "Test failed!"
assert updated_test_vocabulary == {'_','d','e','er','er_','i','l','lo','low','low_','n','ne','new','newer_','o','r','s','t','w'}, "Test failed!"
assert test_merge_rules == [('e r', 'er'), ('er _', 'er_'), ('n e', 'ne'), ('ne w', 'new'),
                            ('l o', 'lo'), ('lo w', 'low'), ('new er_', 'newer_'), ('low _', 'low_')]
print("Success!")


Success!


# Training
learning the merge rules on the Hamlet text data.

In [ ]:
updated_data, updated_vocabulary, merge_rules = bpe(data, vocabulary, 400)


# Tokenization

Constructing tokenizer, leveraging the merge rules acquired in the previous section.

In [ ]:
def token_segmenter(text, merge_rules):
    """
    Tokenize the input text using BPE and the merge rules.

    Parameters:
    - text (str): The input text to be tokenized.
    - merge_rules (list): A list of merging rules obtained from the BPE algorithm.

    Returns:
    - list: A list of tokens obtained using token segmentation rules.
    """
    text = clean_and_normalize_text(text)
    ## START YOU CODE HERE
    tokenized_text = []
    def get_token_keys(text):
        token_frequency, vocabulary = calculate_token_frequencies_and_vocabulary(text)
        return list(token_frequency.keys())

    def get_token_letters(token):
        letters = token.split()
        return letters

    def append_matched(l_idx, letters, letters_list, loop_range, merged_):
        letters_list.append(merged_)
        if l_idx + 2 == loop_range:
            letters_list.append(letters[l_idx+2])

    def append_unmatched(l_idx, letters, letters_list, loop_range):
        letters_list.append(letters[l_idx])
        if l_idx == letters_len - 2:
            letters_list.append(letters[l_idx+1])

    def update_tokenized_text(tokenized_text, tokens):
        for token in tokens:
            letters = token.split()
            tokenized_text.extend(letters)


    splits = get_token_keys(text)

    for space_delimited_pair, merged_form in merge_rules:
        split_idx = 0
        for token in splits:
            token_ = token
            letters = get_token_letters(token)

            if len(letters) == 1:
                split_idx += 1
                continue

            l_idx = 0
            letters_list = []
            loop_range = len(letters) - 1
            letters_len = len(letters)

            while l_idx < loop_range:
                match_window = letters[l_idx] + " " + letters[l_idx+1]

                if match_window == space_delimited_pair:
                    append_matched(l_idx, letters, letters_list, loop_range, merged_form)
                    l_idx += 2
                    continue
                else:
                    append_unmatched(l_idx, letters, letters_list, loop_range)
                l_idx += 1

            token_ = " ".join(letters_list)
            splits[split_idx] = token_
            split_idx += 1

    update_tokenized_text(tokenized_text, splits)

    ## END
    return tokenized_text

In [ ]:
tokenized_text = token_segmenter("that which we call a rose by any other name would smell as sweet", merge_rules)
assert tokenized_text == ['that_', 'which_', 'we_', 'ca', 'll_', 'a_', 'ro',
                          'se_', 'by_', 'an', 'y_', 'other_', 'n', 'a', 'me_',
                          'would_', 's', 'm', 'ell_', 'as_', 'sw', 'eet_'], "Test failed!"
print("Success!")


Success!
